In [16]:
%load_ext autoreload
%autoreload 2

from fee_rules import applies, in_range, capture_matches, txn_mask


def report_task(task_id, level, question, steps, final_status, final_check=None):
    """Display only: print a labelled step-by-step report for one task.

    steps: list of dicts with keys
        desc      - one-line description of the step (from reference_tasks.md)
        result    - value(s) to show; a dict with text keys is shown one line per entry
        checkable - True / False (from reference_tasks.md)
        reason    - short reason for the checkable label
        check     - optional: the assertion this cell already ran for this step
    final_check: optional dict (desc, result, check) for a final-answer assertion.
    Floats are rounded to 6 decimals for display only. Strings are shown as given.
    Call this at the end of the cell, after every assert has run.
    """
    import pandas as pd

    def fmt(x, top=True):
        if isinstance(x, str):
            return x if top else repr(x)
        if isinstance(x, float):
            return repr(round(float(x), 6))
        if isinstance(x, pd.DataFrame):
            return x.round(6).to_string()
        if isinstance(x, pd.Series):
            return (x.round(6) if x.dtype.kind == "f" else x).to_string()
        if isinstance(x, dict):
            return "{" + ", ".join(f"{fmt(k, False)}: {fmt(v, False)}" for k, v in x.items()) + "}"
        if isinstance(x, (list, tuple, set)):
            inner = ", ".join(fmt(v, False) for v in (sorted(x) if isinstance(x, set) else x))
            return {list: "[%s]", tuple: "(%s)", set: "{%s}"}[type(x)] % inner
        return repr(x)

    def show_result(result):
        if isinstance(result, dict) and result and all(isinstance(k, str) for k in result):
            lines = []
            for k, v in result.items():
                text = fmt(v)
                lines += [f"{k}: {text}"] if "\n" not in text else [f"{k}:"] + ["  " + t for t in text.split("\n")]
        else:
            lines = fmt(result).split("\n")
        if len(lines) == 1:
            print("  Result:", lines[0])
        else:
            print("  Result:")
            for line in lines:
                print("    " + line)

    n = len(steps)
    print("=" * 78)
    print(f"TASK {task_id} | level: {level}")
    print(f"Question: {question}")
    print("=" * 78)
    for i, s in enumerate(steps, start=1):
        print(f"[Step {i}/{n}] {s['desc']}")
        show_result(s["result"])
        print(f"  Checkable: {'YES' if s['checkable'] else 'NO'} ({s['reason']})")
        if s.get("check"):
            print(f"  Check: PASS ({s['check']})")
        print()
    if final_check:
        print(f"Final answer check: {final_check['desc']}")
        show_result(final_check["result"])
        print(f"  Check: PASS ({final_check['check']})")
        print()
    k = sum(1 for s in steps if s["checkable"])
    print(f"SUMMARY: {k} of {n} steps checkable | final answer {final_status}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
#task 5: Which issuing country has the highest number of transactions?

import pandas as pd
df = pd.read_csv("../../data/dabstep/data/context/payments.csv")

# step 1: load (shape shown in the report below)

# step 2: count per country
counts = df["issuing_country"].value_counts()

# step 3: answer, should be NL (counts.idxmax(), shown in the report below)

# final answer check: compare with the public answer
expected = "NL"
assert counts.idxmax() == expected

report_task(
    task_id=5,
    level="easy",
    question="Which issuing country has the highest number of transactions?",
    steps=[
        {"desc": "Load payments.csv into a dataframe",
         "result": {"shape": df.shape},
         "checkable": True, "reason": "single deterministic value: shape"},
        {"desc": "Count transactions per issuing_country",
         "result": counts,
         "checkable": True, "reason": "single deterministic value: counts"},
        {"desc": "Take the country with the highest count",
         "result": counts.idxmax(),
         "checkable": True, "reason": "single deterministic value: country code"},
    ],
    final_status="reproduced",
    final_check={"desc": "result vs public answer",
                 "result": {"Result": counts.idxmax(), "Public answer": expected},
                 "check": "assert counts.idxmax() == expected"},
)

TASK 5 | level: easy
Question: Which issuing country has the highest number of transactions?
[Step 1/3] Load payments.csv into a dataframe
  Result: shape: (138236, 21)
  Checkable: YES (single deterministic value: shape)

[Step 2/3] Count transactions per issuing_country
  Result:
    issuing_country
    NL    29622
    IT    28329
    BE    23040
    SE    21716
    FR    14175
    LU     7171
    ES     7109
    GR     7074
  Checkable: YES (single deterministic value: counts)

[Step 3/3] Take the country with the highest count
  Result: NL
  Checkable: YES (single deterministic value: country code)

Final answer check: result vs public answer
  Result:
    Result: NL
    Public answer: NL
  Check: PASS (assert counts.idxmax() == expected)

SUMMARY: 3 of 3 steps checkable | final answer reproduced


In [18]:
#task 49: What is the top country (ip_country) for fraud? A. NL, B. BE, C. ES, D. FR

# step 1: load (reuses df from the task 5 cell; shape shown in the report below)

# step 2: fraud rows
f = df[df["has_fraudulent_dispute"]]

# step 3: fraud transaction count per country
by_count = f["ip_country"].value_counts()

# step 4: fraud euro volume per country
by_volume = f.groupby("ip_country")["eur_amount"].sum()

# step 5: total euro volume per country (display only; by_rate below is computed exactly as before)
total_volume = df.groupby("ip_country")["eur_amount"].sum()

# step 6: fraud rate per country = step 4 / step 5
by_rate = by_volume / df.groupby("ip_country")["eur_amount"].sum()

# step 7: choose the fraud definition (interpretation, no code)

# step 8: top country among the answer options, mapped to its option letter
options = {"A": "NL", "B": "BE", "C": "ES", "D": "FR"}
top_country = by_rate[list(options.values())].idxmax()
top_option = next(letter for letter, country in options.items() if country == top_country)

# final answer check: compare with the public answer
expected = "B. BE"
assert f"{top_option}. {top_country}" == expected

report_task(
    task_id=49,
    level="easy",
    question="What is the top country (ip_country) for fraud? A. NL, B. BE, C. ES, D. FR",
    steps=[
        {"desc": "Load payments.csv",
         "result": {"shape": df.shape},
         "checkable": True, "reason": "single deterministic value: shape"},
        {"desc": "Filter rows where has_fraudulent_dispute is True",
         "result": {"Fraud rows": len(f)},
         "checkable": True, "reason": "single deterministic value: count"},
        {"desc": "Fraud transaction count per ip_country",
         "result": by_count,
         "checkable": True, "reason": "single deterministic value: counts"},
        {"desc": "Fraud euro volume per ip_country (sum of eur_amount)",
         "result": by_volume,
         "checkable": True, "reason": "deterministic numbers"},
        {"desc": "Total euro volume per ip_country (all rows)",
         "result": total_volume,
         "checkable": True, "reason": "deterministic numbers"},
        {"desc": "Fraud rate per country = step 4 / step 5",
         "result": by_rate,
         "checkable": True, "reason": "deterministic numbers"},
        {"desc": "Choose the fraud definition (count vs volume vs rate)",
         "result": "no computed value (definition choice: rate, per manual section 7)",
         "checkable": False,
         "reason": "a definitional decision, validated only by the manual and the final answer"},
        {"desc": "Pick the top country among NL/BE/ES/FR and map to option letter",
         "result": f"{top_option}. {top_country}",
         "checkable": True, "reason": "single deterministic value: option"},
    ],
    final_status="reproduced",
    final_check={"desc": "result vs public answer",
                 "result": {"Result": f"{top_option}. {top_country}", "Public answer": expected},
                 "check": 'assert f"{top_option}. {top_country}" == expected'},
)

TASK 49 | level: easy
Question: What is the top country (ip_country) for fraud? A. NL, B. BE, C. ES, D. FR
[Step 1/8] Load payments.csv
  Result: shape: (138236, 21)
  Checkable: YES (single deterministic value: shape)

[Step 2/8] Filter rows where has_fraudulent_dispute is True
  Result: Fraud rows: 10765
  Checkable: YES (single deterministic value: count)

[Step 3/8] Fraud transaction count per ip_country
  Result:
    ip_country
    NL    2955
    BE    2493
    IT    1652
    SE    1627
    FR     843
    LU     410
    ES     407
    GR     378
  Checkable: YES (single deterministic value: counts)

[Step 4/8] Fraud euro volume per ip_country (sum of eur_amount)
  Result:
    ip_country
    BE    263833.85
    ES     43531.87
    FR     89135.03
    GR     39916.73
    IT    182231.72
    LU     44628.44
    NL    329134.08
    SE    169937.79
  Checkable: YES (deterministic numbers)

[Step 5/8] Total euro volume per ip_country (all rows)
  Result:
    ip_country
    BE    2150473

In [19]:
#task 1305: For account type H and the MCC description: Eating Places and Restaurants, what would be the average fee that the card scheme GlobalCard would charge for a transaction value of 10 EUR?

import json
import pandas as pd

BASE = "../../data/dabstep/data/context/"
mcc_df = pd.read_csv(BASE + "merchant_category_codes.csv")
with open(BASE + "fees.json") as f:
    fees = json.load(f)

# step 1: MCC lookup
hits = mcc_df[mcc_df["description"] == "Eating Places and Restaurants"]
assert len(hits) == 1, hits
mcc = int(hits["mcc"].iloc[0])


# step 2: matching rules
matching = [
    r for r in fees
    if r["card_scheme"] == "GlobalCard"
    and applies(r["account_type"], "H")
    and applies(r["merchant_category_code"], mcc)
]

# step 3: fee per rule for a 10 EUR transaction
value = 10
per_rule = {r["ID"]: r["fixed_amount"] + r["rate"] * value / 10000 for r in matching}

# step 4: average over matching rules
# (step 5 in reference_tasks.md, the unweighted-mean interpretation, has no code)
avg = sum(per_rule.values()) / len(per_rule)
assert f"{avg:.6f}" == "0.123217"


report_task(
    task_id=1305,
    level="hard",
    question="For account type H and the MCC description: Eating Places and Restaurants, what would be "
             "the average fee that the card scheme GlobalCard would charge for a transaction value of 10 EUR?",
    steps=[
        {"desc": 'Look up the MCC code for "Eating Places and Restaurants" in merchant_category_codes.csv',
         "result": {"MCC": mcc},
         "checkable": True, "reason": "single deterministic value: code",
         "check": "exactly one matching row: assert len(hits) == 1"},
        {"desc": "Filter fees.json rules: card_scheme == GlobalCard, account_type contains H (or wildcard), "
                 "MCC contains 5812 (or wildcard)",
         "result": {"Matching rules": len(matching), "Rule IDs": sorted(r["ID"] for r in matching)},
         "checkable": True, "reason": "compare as a set"},
        {"desc": "Compute the fee per matching rule for a 10 EUR transaction",
         "result": {"Fee per rule ID": per_rule},
         "checkable": True, "reason": "tolerance 1e-9"},
        {"desc": "Average the 46 fees",
         "result": {"Average fee": avg},
         "checkable": True, "reason": "single deterministic value: number",
         "check": 'assert f"{avg:.6f}" == "0.123217" (0.123217 is also the public answer)'},
        {"desc": 'Interpret "average fee" as the unweighted mean over matching rules (not volume-weighted)',
         "result": "no computed value (interpretation)",
         "checkable": False, "reason": "an interpretation, validated only by the public answer"},
    ],
    final_status="reproduced",
)

TASK 1305 | level: hard
Question: For account type H and the MCC description: Eating Places and Restaurants, what would be the average fee that the card scheme GlobalCard would charge for a transaction value of 10 EUR?
[Step 1/5] Look up the MCC code for "Eating Places and Restaurants" in merchant_category_codes.csv
  Result: MCC: 5812
  Checkable: YES (single deterministic value: code)
  Check: PASS (exactly one matching row: assert len(hits) == 1)

[Step 2/5] Filter fees.json rules: card_scheme == GlobalCard, account_type contains H (or wildcard), MCC contains 5812 (or wildcard)
  Result:
    Matching rules: 46
    Rule IDs: [5, 38, 92, 114, 140, 141, 160, 162, 192, 204, 221, 257, 267, 276, 280, 319, 325, 357, 359, 403, 427, 428, 456, 477, 498, 513, 556, 572, 612, 660, 666, 682, 688, 704, 709, 725, 741, 792, 813, 861, 888, 891, 892, 899, 917, 921]
  Checkable: YES (compare as a set)

[Step 3/5] Compute the fee per matching rule for a 10 EUR transaction
  Result: Fee per rule ID: {5: 

In [20]:
#task 1753: What are the applicable fee IDs for Belles_cookbook_store in March 2023?

import json
import pandas as pd

BASE = "../../data/dabstep/data/context/"
MERCHANT = "Belles_cookbook_store"

# step 1: load data and look up the merchant
df = pd.read_csv(BASE + "payments.csv")
with open(BASE + "merchant_data.json") as f:
    merchant_data = json.load(f)
with open(BASE + "fees.json") as f:
    fees = json.load(f)

merchant = next(m for m in merchant_data if m["merchant"] == MERCHANT)

# step 2: March 2023 transactions (2023 is not a leap year: March = day_of_year 60-90)
march = df[
    (df["merchant"] == MERCHANT)
    & (df["year"] == 2023)
    & (df["day_of_year"].between(60, 90))
].copy()
march["intracountry"] = march["issuing_country"] == march["acquirer_country"]

# merchant-level attributes (part of step 1 in reference_tasks.md)
account_type = merchant["account_type"]
capture_delay = str(merchant["capture_delay"])
mcc = int(merchant["merchant_category_code"])

# step 3: monthly volume (EUR) and fraud ratio (fraud volume / total volume, in %)
monthly_volume = march["eur_amount"].sum()
fraud_volume = march.loc[march["has_fraudulent_dispute"], "eur_amount"].sum()
fraud_level = 100 * fraud_volume / monthly_volume

# step 4: rules matching the merchant-level conditions
merchant_rules = [
    r for r in fees
    if applies(r["account_type"], account_type)
    and applies(r["merchant_category_code"], mcc)
    and capture_matches(r["capture_delay"], capture_delay)
    and in_range(r["monthly_volume"], monthly_volume)
    and in_range(r["monthly_fraud_level"], fraud_level)
]

# step 5: keep rules that match at least one March transaction
applicable = [r["ID"] for r in merchant_rules if txn_mask(r, march).any()]

# final answer check: compare with the public answer (set comparison, order does not matter)
# (step 6 in reference_tasks.md is the interpretation step and has no code)
expected = {384, 394, 276, 150, 536, 286, 163, 36, 680, 939, 428, 813, 556, 51, 53,
            572, 960, 64, 709, 454, 595, 725, 473, 347, 477, 608, 868, 741, 231, 107,
            626, 249, 123, 381}
assert set(applicable) == expected


report_task(
    task_id=1753,
    level="hard",
    question="What are the applicable fee IDs for Belles_cookbook_store in March 2023?",
    steps=[
        {"desc": "Look up the merchant in merchant_data.json",
         "result": {"Merchant": merchant, "Account type": account_type,
                    "Capture delay": capture_delay, "MCC": mcc},
         "checkable": True, "reason": "deterministic record lookup"},
        {"desc": "Filter payments to the merchant, year 2023, March (day_of_year 60-90, non-leap year)",
         "result": {"March transactions": len(march)},
         "checkable": True, "reason": "single deterministic value: count"},
        {"desc": "Compute March monthly volume (sum of eur_amount) and fraud level "
                 "(fraud volume / total volume, in %)",
         "result": {"March volume (EUR)": monthly_volume, "March fraud level (%)": fraud_level},
         "checkable": True, "reason": "deterministic numbers"},
        {"desc": "Filter fee rules on merchant-level and monthly conditions (null/empty = wildcard)",
         "result": {"Matching rules": len(merchant_rules),
                    "Rule IDs": sorted(r["ID"] for r in merchant_rules)},
         "checkable": True, "reason": "compare as a set"},
        {"desc": "Keep rules matching at least one March transaction "
                 "(card_scheme, is_credit, aci, intracountry)",
         "result": {"Applicable rules": len(applicable), "Rule IDs": sorted(applicable)},
         "checkable": True, "reason": "compare as a set"},
        {"desc": 'Interpret "applicable" (merchant-level match + at least one matching transaction; '
                 "refused transactions included)",
         "result": "no computed value (interpretation)",
         "checkable": False, "reason": "interpretation, validated only by the public answer"},
    ],
    final_status="reproduced",
    final_check={"desc": "applicable fee IDs vs public answer (set comparison, order does not matter)",
                 "result": {"Missing from our list": sorted(expected - set(applicable)),
                            "Extra in our list": sorted(set(applicable) - expected)},
                 "check": "assert set(applicable) == expected"},
)

TASK 1753 | level: hard
Question: What are the applicable fee IDs for Belles_cookbook_store in March 2023?
[Step 1/6] Look up the merchant in merchant_data.json
  Result:
    Merchant: {'merchant': 'Belles_cookbook_store', 'capture_delay': '1', 'acquirer': ['lehman_brothers'], 'merchant_category_code': 5942, 'account_type': 'R'}
    Account type: R
    Capture delay: 1
    MCC: 5942
  Checkable: YES (deterministic record lookup)

[Step 2/6] Filter payments to the merchant, year 2023, March (day_of_year 60-90, non-leap year)
  Result: March transactions: 1277
  Checkable: YES (single deterministic value: count)

[Step 3/6] Compute March monthly volume (sum of eur_amount) and fraud level (fraud volume / total volume, in %)
  Result:
    March volume (EUR): 116436.32
    March fraud level (%): 10.248769
  Checkable: YES (deterministic numbers)

[Step 4/6] Filter fee rules on merchant-level and monthly conditions (null/empty = wildcard)
  Result:
    Matching rules: 47
    Rule IDs: [36, 5

In [21]:
# task 1871: In January 2023 what delta would Belles_cookbook_store pay
# if the relative fee of the fee with ID=384 changed to 1?

import json
import pandas as pd

BASE = "../../data/dabstep/data/context/"
MERCHANT = "Belles_cookbook_store"
FEE_ID = 384

# step 1: load data
df = pd.read_csv(BASE + "payments.csv")

with open(BASE + "merchant_data.json") as f:
    merchant_data = json.load(f)

with open(BASE + "fees.json") as f:
    fees = json.load(f)

merchant = next(m for m in merchant_data if m["merchant"] == MERCHANT)
fee = next(r for r in fees if r["ID"] == FEE_ID)


# step 2: January 2023 transactions
january = df[
    (df["merchant"] == MERCHANT)
    & (df["year"] == 2023)
    & (df["day_of_year"].between(1, 31))
].copy()

january["intracountry"] = (
    january["issuing_country"] == january["acquirer_country"]
)


# step 3: monthly volume and fraud level (merchant attributes are shown under step 1)
account_type = merchant["account_type"]
capture_delay = str(merchant["capture_delay"])
mcc = int(merchant["merchant_category_code"])

monthly_volume = january["eur_amount"].sum()

fraud_volume = january.loc[
    january["has_fraudulent_dispute"],
    "eur_amount"
].sum()

fraud_level = 100 * fraud_volume / monthly_volume


# step 4: check whether Fee 384 matches merchant/month conditions
merchant_rules = [
    r for r in fees
    if applies(r["account_type"], account_type)
    and applies(r["merchant_category_code"], mcc)
    and capture_matches(r["capture_delay"], capture_delay)
    and in_range(r["monthly_volume"], monthly_volume)
    and in_range(r["monthly_fraud_level"], fraud_level)
]

matching_ids = sorted(r["ID"] for r in merchant_rules)


fee384_merchant_match = FEE_ID in matching_ids

# The question explicitly names Fee 384
fee384 = fee

# step 5: find January transactions to which Fee 384 applies
mask = txn_mask(fee384, january)
affected = january.loc[mask].copy()


# step 6: calculate original and counterfactual fees
old_rate = fee384["rate"]
new_rate = 1

fixed_amount = fee384["fixed_amount"]

affected["old_fee"] = (
    fixed_amount
    + old_rate * affected["eur_amount"] / 10000
)

affected["new_fee"] = (
    fixed_amount
    + new_rate * affected["eur_amount"] / 10000
)

old_total = affected["old_fee"].sum()
new_total = affected["new_fee"].sum()

delta = new_total - old_total


# cross-check: fixed_amount cancels out
affected_volume = affected["eur_amount"].sum()

delta_closed_form = (
    (new_rate - old_rate)
    * affected_volume
    / 10000
)


assert abs(delta - delta_closed_form) < 1e-9


# final answer check: compare with public answer
# (step 7 in reference_tasks.md is the interpretation step and has no code)
expected = -0.94810300000017


assert abs(delta - expected) < 1e-9


report_task(
    task_id=1871,
    level="hard",
    question="In January 2023 what delta would Belles_cookbook_store pay "
             "if the relative fee of the fee with ID=384 changed to 1?",
    steps=[
        {"desc": "Load merchant record and fee rule 384",
         "result": {"Merchant": merchant, "Fee rule": "\n".join(f"{k}: {v!r}" for k, v in fee.items()),
                    "Account type": account_type,
                    "Capture delay": capture_delay, "MCC": mcc},
         "checkable": True, "reason": "deterministic record lookup"},
        {"desc": "Filter payments to the merchant, year 2023, January (day_of_year 1-31)",
         "result": {"January transactions": len(january)},
         "checkable": True, "reason": "single deterministic value: count"},
        {"desc": "Compute January monthly volume and fraud level (fraud volume / total volume, in %)",
         "result": {"January volume (EUR)": monthly_volume, "January fraud level (%)": fraud_level},
         "checkable": True, "reason": "deterministic numbers"},
        {"desc": "Check that rule 384 matches the merchant/month conditions",
         "result": {"Merchant/month matching rules": len(matching_ids), "Rule IDs": matching_ids,
                    "Fee 384 matches merchant/month conditions": fee384_merchant_match},
         "checkable": True, "reason": "deterministic set and True/False value"},
        {"desc": "Find January transactions that rule 384 applies to (NexPay, credit, ACI in {C, B})",
         "result": {"Fee 384 affected transactions": len(affected),
                    "Affected transaction volume (EUR)": affected_volume},
         "checkable": True, "reason": "deterministic count and number"},
        {"desc": "Compute old and new fee totals over those transactions (rate 14 -> 1) and the delta",
         "result": {"Original rate": old_rate, "New rate": new_rate,
                    "Original total fee": old_total, "New total fee": new_total,
                    "Delta (14 decimals)": f"{delta:.14f}", "Delta (closed form, 14 decimals)": f"{delta_closed_form:.14f}"},
         "checkable": True, "reason": "tolerance 1e-9",
         "check": "closed-form cross-check: assert abs(delta - delta_closed_form) < 1e-9"},
        {"desc": 'Interpret the question ("relative fee" = rate; delta = new - old; only the affected transactions)',
         "result": "no computed value (interpretation)",
         "checkable": False, "reason": "interpretation, validated only by the public answer"},
    ],
    final_status="reproduced",
    final_check={"desc": "delta vs public answer, tolerance 1e-9",
                 "result": {"Delta (14 decimals)": f"{delta:.14f}",
                            "Public answer (14 decimals)": f"{expected:.14f}"},
                 "check": "assert abs(delta - expected) < 1e-9"},
)

TASK 1871 | level: hard
Question: In January 2023 what delta would Belles_cookbook_store pay if the relative fee of the fee with ID=384 changed to 1?
[Step 1/7] Load merchant record and fee rule 384
  Result:
    Merchant: {'merchant': 'Belles_cookbook_store', 'capture_delay': '1', 'acquirer': ['lehman_brothers'], 'merchant_category_code': 5942, 'account_type': 'R'}
    Fee rule:
      ID: 384
      card_scheme: 'NexPay'
      account_type: []
      capture_delay: None
      monthly_fraud_level: None
      monthly_volume: None
      merchant_category_code: []
      is_credit: True
      aci: ['C', 'B']
      fixed_amount: 0.05
      rate: 14
      intracountry: None
    Account type: R
    Capture delay: 1
    MCC: 5942
  Checkable: YES (deterministic record lookup)

[Step 2/7] Filter payments to the merchant, year 2023, January (day_of_year 1-31)
  Result: January transactions: 1201
  Checkable: YES (single deterministic value: count)

[Step 3/7] Compute January monthly volume and fra

In [22]:
#task 2697: preferred ACI for Belles_cookbook_store's fraudulent January transactions (lowest fees)
import json
import numpy as np
import pandas as pd

BASE = "../../data/dabstep/data/context/"
MERCHANT = "Belles_cookbook_store"
CANDIDATE_ACIS = list("ABCDEFG")  # manual section 4
MODE = "mean"  # how to combine fees when several rules match one transaction: "mean" or "min"

# load data (comes before step 1 of reference_tasks.md)
df = pd.read_csv(BASE + "payments.csv")
with open(BASE + "merchant_data.json") as fh:
    merchant_data = json.load(fh)
with open(BASE + "fees.json") as fh:
    fees = json.load(fh)
merchant = next(m for m in merchant_data if m["merchant"] == MERCHANT)

# step 1: January 2023 transactions
january = df[
    (df["merchant"] == MERCHANT) & (df["year"] == 2023) & (df["day_of_year"].between(1, 31))
].copy()
january["intracountry"] = january["issuing_country"] == january["acquirer_country"]

# step 2: merchant-level and monthly attributes (from the ORIGINAL data)
account_type = merchant["account_type"]
capture_delay = str(merchant["capture_delay"])
mcc = int(merchant["merchant_category_code"])
monthly_volume = january["eur_amount"].sum()
fraud_volume = january.loc[january["has_fraudulent_dispute"], "eur_amount"].sum()
fraud_level = 100 * fraud_volume / monthly_volume

# step 3: rules matching the merchant/month conditions
merchant_rules = [
    r for r in fees
    if applies(r["account_type"], account_type)
    and applies(r["merchant_category_code"], mcc)
    and capture_matches(r["capture_delay"], capture_delay)
    and in_range(r["monthly_volume"], monthly_volume)
    and in_range(r["monthly_fraud_level"], fraud_level)
]

# step 4: fraudulent January transactions
fraud = january[january["has_fraudulent_dispute"]].copy()

# step 5: total fee for the fraudulent transactions under each candidate ACI
def fees_for_aci(txns, aci):
    t = txns.copy()
    t["aci"] = aci
    amounts = t["eur_amount"].to_numpy()
    fee_sum = np.zeros(len(t))
    fee_min = np.full(len(t), np.inf)
    n_match = np.zeros(len(t), dtype=int)
    for r in merchant_rules:
        m = txn_mask(r, t).to_numpy()
        fee = r["fixed_amount"] + r["rate"] * amounts / 10000
        fee_sum += np.where(m, fee, 0)
        fee_min = np.where(m, np.minimum(fee_min, fee), fee_min)
        n_match += m
    has_rule = n_match > 0
    return {
        "mean": (fee_sum[has_rule] / n_match[has_rule]).sum(),
        "min": fee_min[has_rule].sum(),
        "unmatched_txns": int((~has_rule).sum()),
        "avg_rules_per_txn": n_match.mean(),
    }

results = pd.DataFrame({aci: fees_for_aci(fraud, aci) for aci in CANDIDATE_ACIS}).T

# step 6: lowest-total ACI, compared with the ACI of the public answer E:13.57
best = results[MODE].astype(float).idxmin()
assert best == "E"

# step 7: fee for that ACI under this convention, compared with the public answer's 13.57
best_fee = float(results.loc[best, MODE])
fee_reproduced = abs(best_fee - 13.57) < 0.005

report_task(
    task_id=2697,
    level="hard",
    question="For Belles_cookbook_store in January, if we were to move the fraudulent transactions towards "
             "a different ACI, what would be the preferred choice considering the lowest possible fees?",
    steps=[
        {"desc": "Filter payments to the merchant, year 2023, January (day_of_year 1-31)",
         "result": {"January transactions": len(january)},
         "checkable": True, "reason": "single deterministic value: count"},
        {"desc": "Compute January volume and fraud level",
         "result": {"January volume (EUR)": monthly_volume, "January fraud level (%)": fraud_level},
         "checkable": True, "reason": "deterministic numbers"},
        {"desc": "Filter fee rules on merchant-level and monthly conditions",
         "result": {"Merchant/month matching rules": len(merchant_rules)},
         "checkable": True, "reason": "single deterministic value: count"},
        {"desc": "Select fraudulent January transactions (has_fraudulent_dispute)",
         "result": {"Fraudulent transactions": len(fraud),
                    "Fraudulent volume (EUR)": fraud["eur_amount"].sum(),
                    "ACI of those transactions": fraud["aci"].value_counts()},
         "checkable": True, "reason": "deterministic count, number and set"},
        {"desc": "Compute total fees for the fraudulent transactions under each candidate ACI (A-G)",
         "result": results,
         "checkable": False,
         "reason": "convention: when several rules match a transaction their fees are averaged; "
                   "transactions matching no rule add nothing to the total"},
        {"desc": "Choose the ACI with the lowest total fee",
         "result": {f"Lowest-total ACI (mode={MODE})": best},
         "checkable": True, "reason": "deterministic given the step 5 convention",
         "check": 'assert best == "E"'},
        {"desc": "Report the associated fee",
         "result": f"{best_fee:.2f} under this convention; public answer 13.57 "
                   + ("reproduced" if fee_reproduced else "NOT REPRODUCED"),
         "checkable": False, "reason": "fee value of the public answer not reproduced"},
    ],
    final_status="partly reproduced",
)

TASK 2697 | level: hard
Question: For Belles_cookbook_store in January, if we were to move the fraudulent transactions towards a different ACI, what would be the preferred choice considering the lowest possible fees?
[Step 1/7] Filter payments to the merchant, year 2023, January (day_of_year 1-31)
  Result: January transactions: 1201
  Checkable: YES (single deterministic value: count)

[Step 2/7] Compute January volume and fraud level
  Result:
    January volume (EUR): 113260.42
    January fraud level (%): 10.313064
  Checkable: YES (deterministic numbers)

[Step 3/7] Filter fee rules on merchant-level and monthly conditions
  Result: Merchant/month matching rules: 47
  Checkable: YES (single deterministic value: count)

[Step 4/7] Select fraudulent January transactions (has_fraudulent_dispute)
  Result:
    Fraudulent transactions: 94
    Fraudulent volume (EUR): 11680.62
    ACI of those transactions:
      aci
      G    94
  Checkable: YES (deterministic count, number and set)



In [23]:
# task 2697 diagnostic 1: drop one transaction-level condition at a time
print("DIAGNOSTIC 1 (outside the 7 steps): per-transaction fee totals with one transaction-level "
      "condition dropped at a time. No variant gives 13.57.")
def mask_variant(rule, t, ignore=()):
    m = pd.Series(True, index=t.index)
    if "scheme" not in ignore and rule["card_scheme"]:
        m &= t["card_scheme"] == rule["card_scheme"]
    if "credit" not in ignore and rule["is_credit"] is not None:
        m &= t["is_credit"] == rule["is_credit"]
    if "aci" not in ignore and rule["aci"]:
        m &= t["aci"].isin(rule["aci"])
    if "intra" not in ignore and rule["intracountry"] is not None:
        m &= t["intracountry"] == bool(rule["intracountry"])
    return m.to_numpy()

def variant_totals(txns, aci, ignore=()):
    t = txns.copy()
    t["aci"] = aci
    amounts = t["eur_amount"].to_numpy()
    fee_sum = np.zeros(len(t))
    fee_min = np.full(len(t), np.inf)
    n_match = np.zeros(len(t), dtype=int)
    for r in merchant_rules:
        m = mask_variant(r, t, ignore)
        fee = r["fixed_amount"] + r["rate"] * amounts / 10000
        fee_sum += np.where(m, fee, 0)
        fee_min = np.where(m, np.minimum(fee_min, fee), fee_min)
        n_match += m
    ok = n_match > 0
    return {"mean": (fee_sum[ok] / n_match[ok]).sum(), "min": fee_min[ok].sum()}

variants = {
    "all conditions (baseline)": (),
    "ignore card_scheme": ("scheme",),
    "ignore is_credit": ("credit",),
    "ignore intracountry": ("intra",),
    "ignore scheme+credit": ("scheme", "credit"),
    "ACI only": ("scheme", "credit", "intra"),
}
rows = []
for name, ignore in variants.items():
    per_aci = {a: variant_totals(fraud, a, ignore) for a in CANDIDATE_ACIS}
    for mode in ("mean", "min"):
        vals = {a: per_aci[a][mode] for a in CANDIDATE_ACIS}
        best_v = min(vals, key=vals.get)
        rows.append((name, mode, best_v, round(vals[best_v], 2), round(vals["E"], 2)))
out = pd.DataFrame(rows, columns=["variant", "mode", "best_aci", "best_total", "E_total"])
print(out.to_string(index=False))

DIAGNOSTIC 1 (outside the 7 steps): per-transaction fee totals with one transaction-level condition dropped at a time. No variant gives 13.57.
                  variant mode best_aci  best_total  E_total
all conditions (baseline) mean        E       41.66    41.66
all conditions (baseline)  min        E       16.63    16.63
       ignore card_scheme mean        E       65.91    65.91
       ignore card_scheme  min        B       19.29    21.74
         ignore is_credit mean        E       59.73    59.73
         ignore is_credit  min        E       30.79    30.79
      ignore intracountry mean        E       41.66    41.66
      ignore intracountry  min        E       16.63    16.63
     ignore scheme+credit mean        C       65.38    70.68
     ignore scheme+credit  min        C       15.25    21.74
                 ACI only mean        C       60.96    65.81
                 ACI only  min        C       14.61    21.73


In [24]:
# task 2697 diagnostic 2: aggregate-volume hypothesis
# treat the total fraud volume as ONE transaction value, as task 1305 did with 10 EUR
print("DIAGNOSTIC 2 (outside the 7 steps): total fraud volume treated as one transaction value, with "
      "and without card_scheme / is_credit conditions. No variant gives 13.57.")

V = fraud["eur_amount"].sum()
schemes_present = set(fraud["card_scheme"].unique())
credit_present = set(fraud["is_credit"].unique())
print("Fraud volume:", V, "| schemes:", schemes_present, "| is_credit values:", credit_present)


def rules_for(aci, use_scheme=False, use_credit=False):
    rs = [r for r in merchant_rules if applies(r["aci"], aci)]
    if use_scheme:
        rs = [r for r in rs if r["card_scheme"] is None or r["card_scheme"] in schemes_present]
    if use_credit:
        rs = [r for r in rs if r["is_credit"] is None or r["is_credit"] in credit_present]
    return rs


variants = {
    "ACI only": (False, False),
    "ACI + scheme": (True, False),
    "ACI + is_credit": (False, True),
    "ACI + scheme + is_credit": (True, True),
}

rows = []
for name, (us, uc) in variants.items():
    totals = {}
    for aci in CANDIDATE_ACIS:
        fs = [r["fixed_amount"] + r["rate"] * V / 10000 for r in rules_for(aci, us, uc)]
        if fs:
            totals[aci] = {"mean": sum(fs) / len(fs), "min": min(fs), "n_rules": len(fs)}
    for mode in ("mean", "min"):
        vals = {a: t[mode] for a, t in totals.items()}
        best = min(vals, key=vals.get)
        rows.append((name, mode, best, round(vals[best], 2), round(vals.get("E", float("nan")), 2),
                     totals.get("E", {}).get("n_rules", 0)))

out2 = pd.DataFrame(rows, columns=["variant", "mode", "best_aci", "best_total", "E_total", "E_rules"])
print(out2.to_string(index=False))
print("\nMatches E:13.57:")
print(out2[(out2["best_aci"] == "E") & (out2["best_total"] == 13.57)])

DIAGNOSTIC 2 (outside the 7 steps): total fraud volume treated as one transaction value, with and without card_scheme / is_credit conditions. No variant gives 13.57.
Fraud volume: 11680.62 | schemes: {'NexPay', 'TransactPlus', 'SwiftCharge', 'GlobalCard'} | is_credit values: {np.True_}
                 variant mode best_aci  best_total  E_total  E_rules
                ACI only mean        C       54.28    60.63        7
                ACI only  min        C       11.72    19.88        7
            ACI + scheme mean        C       54.28    60.63        7
            ACI + scheme  min        C       11.72    19.88        7
         ACI + is_credit mean        E       60.02    60.02        3
         ACI + is_credit  min        B       16.40    19.88        3
ACI + scheme + is_credit mean        E       60.02    60.02        3
ACI + scheme + is_credit  min        B       16.40    19.88        3

Matches E:13.57:
Empty DataFrame
Columns: [variant, mode, best_aci, best_total, E_total, E_